In [1]:
import os
from dotenv import load_dotenv

load_dotenv()
krycklan_data_path = os.getenv('KRYCKLAN_DATA_PATH')
print(f'Krycklan data path: {krycklan_data_path}')

Krycklan data path: /Users/jpnousu/Library/CloudStorage/OneDrive-Luonnonvarakeskus/Krycklan data


In [2]:
import geopandas as gpd
import rasterio
import tools
import numpy as np

### SOIL MAP (FROM ANNELI)

In [3]:
file1 = os.path.join(krycklan_data_path, 'GIS/SOIL/krycklan_QD/krycklan_QD_J1.shp')
file2 = os.path.join(krycklan_data_path, 'GIS/SOIL/krycklan_QD/krycklan_QD_J2.shp')

df1 = gpd.read_file(file1)
df2 = gpd.read_file(file2)
codes = {}
for id in df1['JY1'].unique():
    codes[id] = df1.loc[df1['JY1'] == id, 'JY1_TX'].iloc[0]
for id in df2['JG2'].unique():
    codes[id] = df2.loc[df2['JG2'] == id, 'JG2_TX'].iloc[0]
print(codes)

{np.int64(100): 'Morän', np.int64(84): 'Postglacial sand--grus', np.int64(89): 'Svallsediment, grus--block', np.int64(75): 'Torv', np.int64(50): 'Isälvssediment', np.int64(86): 'Lera--silt', np.int64(200): 'Fyllning', np.int64(31): 'Postglacial sand', np.int64(8809): 'Älvsediment, sand', np.int64(91): 'Vatten', np.int64(888): 'Berg'}


In [4]:
from scipy.ndimage import distance_transform_edt

resos = [20, 40, 80, 160]
id = 0
for reso in resos:
    id += 1
    # Reference file path
    ref_fp = os.path.join(krycklan_data_path, f'GIS/FOREST_BASIC_KRYCKLAN/processed/canopy_height_{reso}.asc')
    out_fp1 = os.path.join(krycklan_data_path, f'GIS/SOIL/krycklan_QD/krycklan_QD_J1_{reso}.asc')
    out_fp2 = os.path.join(krycklan_data_path, f'GIS/SOIL/krycklan_QD/krycklan_QD_J2_{reso}.asc')
    out_fp3 = os.path.join(krycklan_data_path, f'GIS/SOIL/krycklan_QD/krycklan_QD_J1_J2_{reso}.asc')
    out_fp4 = os.path.join(krycklan_data_path, f'GIS/SOIL/krycklan_QD/lake_mask_{reso}.asc')
    out_fp5 = os.path.join(krycklan_data_path, f'GIS/SOIL/krycklan_QD/krycklan_QD_J1_J2_final_{reso}.asc')

    tools.rasterize_shapefile(shapefile=file1, burn_field='JY1', subset=None, out_fp=out_fp1, ref_raster=ref_fp, plot=False, save_in='asc')
    tools.rasterize_shapefile(shapefile=file2, burn_field='JG2', subset=None, out_fp=out_fp2, ref_raster=ref_fp, plot=False, save_in='asc')
    tools.fill_layer_na_with_layer(priority_layer=out_fp1, secondary_layer=out_fp2, out_fp=out_fp3, save_in='asc')
    
    with rasterio.open(out_fp3) as src:
        data = src.read(1)
        meta = src.meta.copy()
        new_data = data.copy()
        new_data[data != 91.0] = meta['nodata']
        new_data[data == 91.0] = 1.0
    with rasterio.open(out_fp4, 'w', **meta) as dst:
        out = dst.write(new_data, 1)

    with rasterio.open(out_fp3) as src:
        data = src.read(1)
        new_data = data.copy()
        meta = src.meta.copy()
        nodata = meta['nodata']
        if id == 1:
            orig_uniq = np.unique(data[data != nodata])
            new_uniq = np.arange(1, len(orig_uniq)+1)
        for i in range(len(orig_uniq)):
            new_data[data == orig_uniq[i]] = new_uniq[i]

    # Fill remaining nodata holes with nearest-neighbor interpolation
    valid_mask = new_data != nodata
    _, indices = distance_transform_edt(~valid_mask, return_indices=True)
    new_data = new_data[tuple(indices)]

    with rasterio.open(out_fp5, 'w', **meta) as dst:
        out = dst.write(new_data, 1)

#### GROUP SOIL TYPES INTO HYDRAULIC CLASSES

Reclassify the sequential soil-type raster (`krycklan_QD_J1_J2_final`) into five groups based on hydraulic conductivity:

| Group | Code | Soil types (deep_id) |
|---|---|---|
| rock   | 1 | Bedrock (10) |
| peat   | 2 | Peat (3) |
| fine   | 3 | Clay/silt (5), Water (7) |
| medium | 4 | Moraine (8) |
| coarse | 5 | Postglacial sand (1,2,4), washed sediment (6), fill (9), fluvial sand (11) |

In [ ]:
# Mapping from deep_id (pixel value in final raster) to hydraulic group:
#   1=rock, 2=peat, 3=fine, 4=medium, 5=coarse
deep_id_to_group = {
    1:  5,   # Postglacial_sand              → coarse
    2:  5,   # Glaciofluvial_sediment         → coarse
    3:  2,   # Peat                           → peat
    4:  5,   # Postglacial_sand_gravel        → coarse
    5:  3,   # Clay_silt                      → fine
    6:  5,   # Washed_sediment_gravel_boulders → coarse
    7:  4,   # Water                          → medium
    8:  4,   # Moraine                        → medium
    9:  5,   # Fill                           → coarse
    10: 1,   # Bedrock                        → rock
    11: 5,   # Fluvial_sediment_sand          → coarse
}

# Build lookup array: index = deep_id, value = group code
max_id = max(deep_id_to_group.keys())
lookup = np.zeros(max_id + 1, dtype=np.int32)
for deep_id, group in deep_id_to_group.items():
    lookup[deep_id] = group

resos = [20, 40, 80, 160]
for reso in resos:
    fp = os.path.join(krycklan_data_path, f'GIS/SOIL/krycklan_QD/krycklan_QD_J1_J2_final_{reso}.asc')
    out_fp = os.path.join(krycklan_data_path, f'GIS/SOIL/krycklan_QD/grouped_soil_{reso}.asc')

    with rasterio.open(fp) as src:
        data = src.read(1).astype(np.int32)
        profile = src.profile
        nodata = int(src.nodata) if src.nodata is not None else -9999

    # Remap valid deep_id values to group codes
    grouped = np.full_like(data, nodata)
    valid = (data >= 1) & (data <= max_id)
    grouped[valid] = lookup[data[valid]]

    profile.update(dtype=rasterio.int32, nodata=nodata)
    with rasterio.open(out_fp, 'w', **profile) as dst:
        dst.write(grouped, 1)

    print(f'{reso} m → {os.path.basename(out_fp)}, groups present: {np.unique(grouped[grouped != nodata])}')


20 m → grouped_soil_20.asc, groups present: [1 2 3 4 5]
40 m → grouped_soil_40.asc, groups present: [1 2 3 4 5]
80 m → grouped_soil_80.asc, groups present: [1 2 3 4 5]
160 m → grouped_soil_160.asc, groups present: [1 2 3 4 5]


#### SOIL DEPTH INFORMATION CROP AND FIT

In [6]:
resos = [20, 40, 80, 160]
id = 0
for reso in resos:
    match = os.path.join(krycklan_data_path, f'GIS/FOREST_BASIC_KRYCKLAN/processed/canopy_height_{reso}.asc')
    fp = os.path.join(krycklan_data_path, 'GIS/SOIL/jorddjupsmodell/jorddjupsmodell_10x10m.tif')
    out_fp = os.path.join(krycklan_data_path, f'GIS/SOIL/jorddjupsmodell/PROCESSED/SOIL_DEPTH_{reso}.asc')
    tools.reproj_match(infile=fp, match=match, outfile=out_fp, resampling_method='average', save_in='asc')

Coregistered to shape: 575 515 
 Affine | 20.00, 0.00, 726726.00|
| 0.00,-20.00, 7138690.00|
| 0.00, 0.00, 1.00|
Coregistered to shape: 287 257 
 Affine | 40.00, 0.00, 726726.00|
| 0.00,-40.00, 7138670.00|
| 0.00, 0.00, 1.00|
Coregistered to shape: 143 128 
 Affine | 80.00, 0.00, 726726.00|
| 0.00,-80.00, 7138630.00|
| 0.00, 0.00, 1.00|
Coregistered to shape: 71 64 
 Affine | 160.00, 0.00, 726726.00|
| 0.00,-160.00, 7138550.00|
| 0.00, 0.00, 1.00|
